### Representación básica: DFA adaptado al comportamiento de compra

| Variable          | Significado                             |
| ----------------- | --------------------------------------- |
| `days_since_previous_order` | Días desde el pedido anterior al último pedido histórico |
| `frequency`       | Número total de pedidos realizados      |
| `avg_basket_size` | Número medio de productos por pedido    |


In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../../data/instacart/")
DATA_DIR_CLEAN = Path("../../data_without_nulls/instacart/")

In [2]:
orders = pd.read_csv(DATA_DIR_CLEAN / "orders.csv")
order_products_prior = pd.read_csv(DATA_DIR / "order_products__prior.csv")

#### Recency

In [3]:
days_since_previous_order = (
    orders[orders["eval_set"].isin(["train", "test"])]
    [["user_id", "days_since_prior_order"]]
    .rename(columns={"days_since_prior_order": "days_since_previous_order"})
)

days_since_previous_order.head()

,user_id,days_since_previous_order
10,1,14.0
25,2,30.0
38,3,11.0
44,4,30.0
49,5,6.0


### Frequency

In [4]:
# Nos quedamos únicamente con los pedidos históricos (prior)
orders_prior = orders[orders["eval_set"] == "prior"].copy()

# Número de pedidos históricos realizados por cada usuario
frequency = (
    orders_prior
    .groupby("user_id")["order_id"]
    .nunique()
    .reset_index(name="frequency")
)

frequency.head()

,user_id,frequency
0,1,10
1,2,14
2,3,12
3,4,5
4,5,4


### Tamaño medio de cesta

In [5]:
# Número de productos incluidos en cada pedido
basket_size = (
    order_products_prior
    .groupby("order_id")
    .size()
    .reset_index(name="basket_size")
)

basket_size.head()

,order_id,basket_size
0,2,9
1,3,8
2,4,13
3,5,26
4,6,3


In [6]:
# Añadimos user_id a cada pedido
basket_size = basket_size.merge(
    orders_prior[["order_id", "user_id"]],
    on="order_id",
    how="left"
)

In [7]:
avg_basket = (
    basket_size
    .groupby("user_id")["basket_size"]
    .mean()
    .reset_index(name="avg_basket_size")
)

avg_basket.head()

,user_id,avg_basket_size
0,1,5.900000
1,2,13.928571
2,3,7.333333
3,4,3.600000
4,5,9.250000


### Unión de las tres variables al dataset

In [8]:
instacart_simple = (
    days_since_previous_order
    .merge(frequency, on="user_id", how="inner")
    .merge(avg_basket, on="user_id", how="inner")
)

instacart_simple.head()

,user_id,days_since_previous_order,frequency,avg_basket_size
0,1,14.0,10,5.900000
1,2,30.0,14,13.928571
2,3,11.0,12,7.333333
3,4,30.0,5,3.600000
4,5,6.0,4,9.250000


In [9]:
print("Dimensiones:", instacart_simple.shape)

print("\nDuplicados por usuario:")
print(instacart_simple["user_id"].duplicated().sum())

print("\nValores nulos:")
print(instacart_simple.isnull().sum())

print("\nEstadísticas descriptivas:")
display(instacart_simple.describe())

Dimensiones: (206209, 4)

Duplicados por usuario:
0

Valores nulos:
user_id            0
days_since_previous_order            0
frequency          0
avg_basket_size    0
dtype: int64

Estadísticas descriptivas:


,user_id,days_since_previous_order,frequency,avg_basket_size
count,206209.000000,206209.000000,206209.000000,206209.000000
mean,103105.000000,17.061782,15.590367,9.951586
std,59527.555167,10.672178,16.654774,5.863570
min,1.000000,0.000000,3.000000,1.000000
25%,51553.000000,7.000000,5.000000,5.740741
50%,103105.000000,15.000000,9.000000,8.933333
75%,154657.000000,30.000000,19.000000,13.000000
max,206209.000000,30.000000,99.000000,70.250000


In [10]:
print("Usuarios originales:")
print(orders["user_id"].nunique())

print("\nUsuarios dataset simple:")
print(instacart_simple["user_id"].nunique())

Usuarios originales:
206209

Usuarios dataset simple:
206209


### Guardado del dataset básico DFA

In [11]:
instacart_simple.to_csv("../../simple_datasets/instacart_model_simple.csv",
    index=False
)